In [1]:
%%bash
git clone -b dev https://github.com/hzk1102/DDColor.git

Cloning into 'DDColor'...


In [2]:
cd DDColor/

/kaggle/working/DDColor


In [3]:
import gdown

# File 2
url2 = "https://drive.google.com/uc?id=1dXda8H1WcxwHvlLZNicjwiRchMUTUXwA"
gdown.download(url2, quiet=False)

print("Download complete!")

Downloading...
From (original): https://drive.google.com/uc?id=1dXda8H1WcxwHvlLZNicjwiRchMUTUXwA
From (redirected): https://drive.google.com/uc?id=1dXda8H1WcxwHvlLZNicjwiRchMUTUXwA&confirm=t&uuid=781dc156-4e76-420b-938a-53e0026760e1
To: /kaggle/working/DDColor/ViCoWDataset.zip
100%|██████████| 651M/651M [00:23<00:00, 27.9MB/s] 

Download complete!


In [4]:
import zipfile

with zipfile.ZipFile('ViCoWDataset.zip', 'r') as zf:
    zf.extractall()

In [5]:
!cp -r "ViCoW A Dataset for Colorization and Restoration of Vietnam War Imagery"/* ViCoW_Dataset/


In [6]:
!pip install --upgrade pip setuptools wheel modelscope
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 113.2 MB/s eta 0:00:0000:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.2.19 which is incompatible.


In [7]:
!pip install lpips opencv-python scikit-image numpy 

In [8]:
!pip install -r requirements312.txt

In [9]:
import torch
torch.__version__
!git config --global --add safe.directory /home/h/DDColor

In [10]:
!pip install -e . --no-build-isolation
#!python setup.py develop

Obtaining file:///kaggle/working/DDColor
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for basicsr (pyproject.toml) ... done
  Created wheel for basicsr: filename=basicsr-1.3.4.6-0.editable-py3-none-any.whl size=10290 sha256=8a16f2f6cff976563d90ee6d4198f51cd946d6819d772dc984dcc346a0884ca4
  Stored in directory: /tmp/pip-ephem-wheel-cache-fc0_5mge/wheels/79/ef/4a/3242a5b4f83e187e90ae0d0607ca87f99f2e1793e1950214c3
Successfully built basicsr


In [13]:
!gdown --id 1g9KtWGbfbMRGP6E0nQHJDD6K05q0BCHd -O ./modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1g9KtWGbfbMRGP6E0nQHJDD6K05q0BCHd
From (redirected): https://drive.google.com/uc?id=1g9KtWGbfbMRGP6E0nQHJDD6K05q0BCHd&confirm=t&uuid=d2a912f1-d91f-4c55-b54f-07a6aeaebdd9
To: /kaggle/working/DDColor/modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt
100%|████████████████████████████████████████| 912M/912M [00:30<00:00, 29.7MB/s]


In [15]:
from modelscope.hub.snapshot_download import snapshot_download

model_dir = snapshot_download('damo/cv_ddcolor_image-colorization', cache_dir='./modelscope')
print('model assets saved to %s' % model_dir)

model assets saved to ./modelscope/damo/cv_ddcolor_image-colorization


In [20]:
torch.cuda.is_available()

True

In [17]:
# chuan bi data
import os
import pandas as pd
import shutil
from tqdm import tqdm

def prepare_data(csv_file, split_name):
    # Tên folder gốc chứa ảnh của bạn
    base_dir = 'ViCoW_Dataset' 
    
    if not os.path.exists(csv_file):
        print(f"Bỏ qua: Không tìm thấy file {csv_file}")
        return

    # Tạo thư mục đích: dataset/train, dataset/val, dataset/test
    output_dir = os.path.join('dataset', split_name)
    os.makedirs(output_dir, exist_ok=True)

    # Đọc CSV
    df = pd.read_csv(csv_file)
    print(f"\n--- Đang xử lý tập {split_name.upper()} ({len(df)} ảnh) ---")
    
    success_count = 0
    missing_samples = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Đường dẫn từ CSV: Color_image/VIDEO2_VaoNamRaBac/frame_0998.jpg
        rel_path = row['colorPath']
        
        # Ghép thành đường dẫn thực tế: ViCoW_Dataset/Color_image/...
        src_path = os.path.join(base_dir, rel_path)
        
        if os.path.exists(src_path):
            # Tách lấy tên VIDEO và tên FRAME để tạo tên file mới (tránh trùng)
            # Ví dụ: VIDEO2_VaoNamRaBac_frame_0998.jpg
            path_parts = rel_path.split('/')
            video_folder = path_parts[1] 
            filename = path_parts[-1]
            new_filename = f"{video_folder}_{filename}"
            
            dst_path = os.path.join(output_dir, new_filename)
            shutil.copy2(src_path, dst_path)
            success_count += 1
        else:
            if len(missing_samples) < 1: # Lưu lại mẫu lỗi đầu tiên để báo cáo
                missing_samples.append(os.path.abspath(src_path))

    print(f"Kết quả: Copy thành công {success_count}/{len(df)} ảnh vào '{output_dir}'")
    
    if success_count == 0 and len(missing_samples) > 0:
        print(f"\n[CẢNH BÁO] Không tìm thấy ảnh nào! Script đã thử tìm ở:")
        print(f" -> {missing_samples[0]}")
        print("Hãy đảm bảo folder 'ViCoW_Dataset' nằm cùng cấp với file script này.")

# --- Chạy script ---
tasks = [('ViCoW_Dataset/train.csv', 'train'), ('ViCoW_Dataset/val.csv', 'val'), ('ViCoW_Dataset/test.csv', 'test')]
for csv, split in tasks:
    prepare_data(csv, split)

print("\nHoàn tất!")


--- Đang xử lý tập TRAIN (1327 ảnh) ---


100%|██████████| 1327/1327 [00:00<00:00, 3043.40it/s]


Kết quả: Copy thành công 1327/1327 ảnh vào 'dataset/train'

--- Đang xử lý tập VAL (187 ảnh) ---


100%|██████████| 187/187 [00:00<00:00, 2856.34it/s]


Kết quả: Copy thành công 187/187 ảnh vào 'dataset/val'

--- Đang xử lý tập TEST (382 ảnh) ---


100%|██████████| 382/382 [00:00<00:00, 3040.95it/s]

Kết quả: Copy thành công 382/382 ảnh vào 'dataset/test'

Hoàn tất!


In [18]:
%%bash
python data_list/get_meta_file.py --output-name ./dataset/train.txt --data-path ./dataset/train
python data_list/get_meta_file.py --output-name ./dataset/val.txt --data-path ./dataset/val
python data_list/get_meta_file.py --output-name ./dataset/test.txt --data-path ./dataset/test

Generating ./dataset/train.txt from ./dataset/train ...
Done.
Generating ./dataset/val.txt from ./dataset/val ...
Done.
Generating ./dataset/test.txt from ./dataset/test ...
Done.


100%|██████████| 382/382 [00:00<00:00, 2122151.16it/s]


In [26]:
!wget https://dl.fbaipublicfiles.com/convnext/convnext_large_22k_224.pth -P pretrain/
!wget https://download.pytorch.org/models/inception_v3_google-1a9a5a14.pth -P pretrain/

--2025-12-30 17:01:29--  https://dl.fbaipublicfiles.com/convnext/convnext_large_22k_224.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 108.158.20.120, 108.158.20.43, 108.158.20.21, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|108.158.20.120|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 919310343 (877M) [binary/octet-stream]
Saving to: ‘pretrain/convnext_large_22k_224.pth’

convnext_large_22k_ 100%[===================>] 876.72M  26.9MB/s    in 34s     

2025-12-30 17:02:03 (26.1 MB/s) - ‘pretrain/convnext_large_22k_224.pth’ saved [919310343/919310343]

--2025-12-30 17:02:03--  https://download.pytorch.org/models/inception_v3_google-1a9a5a14.pth
Resolving download.pytorch.org (download.pytorch.org)... 18.67.110.21, 18.67.110.46, 18.67.110.56, ...
Connecting to download.pytorch.org (download.pytorch.org)|18.67.110.21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 108857766 (104M) [application/oct

In [ ]:
!pip install --upgrade wandb
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter: 


Aborted!
^C


In [2]:
# /home/h/DDColor/options/train/train_ViCoW.yml
!sh scripts/train_kaggle.sh

/home/h/ddcolor312/lib/python3.12/site-packages/torch/distributed/launch.py:208: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  main()
2025-12-31 09:31:54,277 INFO: 
                ____                _       _____  ____
               / __ ) ____ _ _____ (_)_____/ ___/ / __ \
              / __  |/ __ `// ___// // ___/\__ \ / /_/ /
             / /_/ // /_/ /(__  )/ // /__ ___/ // _, _/
            /_____/ \__,_//____//_/ \___//____//_/ |_|
     ______                   __   __                 __      __
    / ____/____   ____   ____/ /  / /   __  __ _____ / /__   / /
   / / __ / __ \ / __ \ / __  /  / /   / / / // ___// //_/  / /
  / /_/ // /_/ // /_/ /

### Tensorboard

In [ ]:
!tensorboard --logdir=tb_logger/train_ViCoW_2 --port=6006

## Infer, gen out_test

In [ ]:
# !python infer.py --model_path ./modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt --input ./assets/test_images

In [ ]:
#!python infer.py --model_path ./modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt --input ViCoW_Dataset/Grayscale_image/VIDEO1_NhungNguoiVietLenHuyenThoai --output out1 --input_size 512 --model_size large
#DDColor/experiments/train_ViCoW/models/net_g_5000.pth

!python infer.py --model_path experiments/train_ViCoW_2/models/net_g_25000.pth \
                --input dataset/test   \
                --output out_test_25000 \
                --input_size 512 # --model_size large

# from infer_hf import DDColorHF

# ddcolor_paper_tiny = DDColorHF.from_pretrained("piddnad/ddcolor_paper_tiny")
# ddcolor_paper      = DDColorHF.from_pretrained("piddnad/ddcolor_paper")
# ddcolor_modelscope = DDColorHF.from_pretrained("piddnad/ddcolor_modelscope")
# ddcolor_artistic   = DDColorHF.from_pretrained("piddnad/ddcolor_artistic")

# python infer_hf.py --model_name ddcolor_artistic --input ViCoW_Dataset/Grayscale_image/VIDEO1_NhungNguoiVietLenHuyenThoai --output out_artistic --input_size 512

# python infer_hf.py --model_path ./modelscope/damo/cv_ddcolor_image-colorization/pytorch_model.pt --input ViCoW_Dataset/Grayscale_image/VIDEO1_NhungNguoiVietLenHuyenThoai --output out1 --input_size 512


## Validate

In [ ]:
import os
import cv2
import torch
import numpy as np
import lpips
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# Khởi tạo LPIPS (VGG) - Yêu cầu: pip install lpips torch
loss_fn_vgg = lpips.LPIPS(net='vgg')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss_fn_vgg.to(device)

def get_colorfulness(image):
    """Tính độ rực rỡ của ảnh (Hasler and Suesstrunk)"""
    (B, G, R) = cv2.split(image.astype("float"))
    rg = np.absolute(R - G)
    yb = np.absolute(0.5 * (R + G) - B)
    std_root = np.sqrt(np.std(rg)**2 + np.std(yb)**2)
    mean_root = np.sqrt(np.mean(rg)**2 + np.mean(yb)**2)
    return std_root + (0.3 * mean_root)

def im2tensor(image):
    """Chuyển ảnh OpenCV sang Tensor cho LPIPS"""
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = torch.from_numpy(image).permute(2, 0, 1).float()
    image = (image / 127.5) - 1.0
    return image.unsqueeze(0).to(device)

def calculate_full_metrics(gt_dir, out_dir):
    gt_images = sorted([f for f in os.listdir(gt_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    
    # Lưu trữ kết quả
    m = {'psnr': [], 'ssim': [], 'lpips': [], 'c_ratio': []}
    
    print(f"--- Đang đánh giá {len(gt_images)} ảnh trên {device} ---")

    for filename in gt_images:
        path_gt = os.path.join(gt_dir, filename)
        path_out = os.path.join(out_dir, filename)

        if not os.path.exists(path_out): continue

        img_gt = cv2.imread(path_gt)
        img_out = cv2.imread(path_out)

        # Đảm bảo cùng kích thước
        if img_gt.shape != img_out.shape:
            img_out = cv2.resize(img_out, (img_gt.shape[1], img_gt.shape[0]))

        # 1. PSNR & SSIM (Độ khớp pixel & cấu trúc)
        m['psnr'].append(psnr(img_gt, img_out, data_range=255))
        m['ssim'].append(ssim(img_gt, img_out, channel_axis=2, data_range=255))

        # 2. LPIPS (Độ chân thực cảm quan)
        t_gt, t_out = im2tensor(img_gt), im2tensor(img_out)
        with torch.no_grad():
            m['lpips'].append(loss_fn_vgg(t_gt, t_out).item())

        # 3. Colorfulness Ratio (Độ đậm nhạt của màu)
        c_gt = get_colorfulness(img_gt)
        c_out = get_colorfulness(img_out)
        m['c_ratio'].append((c_out / c_gt) * 100 if c_gt != 0 else 100)

        print(f"[{filename}] PSNR: {m['psnr'][-1]:.2f} | SSIM: {m['ssim'][-1]:.4f} | LPIPS: {m['lpips'][-1]:.4f} | Color: {m['c_ratio'][-1]:.1f}%")

    if m['psnr']:
        print("\n" + "="*50)
        print(f"KẾT QUẢ TRUNG BÌNH TOÀN BỘ TẬP TEST:")
        print(f"1. PSNR (Độ khớp pixel)   : {np.mean(m['psnr']):.2f} dB ↑")
        print(f"2. SSIM (Độ khớp cấu trúc): {np.mean(m['ssim']):.4f} ↑")
        print(f"3. LPIPS (Độ chân thực AI): {np.mean(m['lpips']):.4f} ↓ (Thấp là tốt)")
        print(f"4. COLOR RATIO (Độ đậm)   : {np.mean(m['c_ratio']):.1f}% (Càng gần 100% càng tốt)")
        print("="*50)
        
        # Đưa ra chẩn đoán
        avg_c = np.mean(m['c_ratio'])
        if avg_c < 85:
            print("CHẨN ĐOÁN: Ảnh đang bị NHẠT. Hãy tăng color_enhance_factor trong config.")
        elif avg_c > 115:
            print("CHẨN ĐOÁN: Ảnh đang bị QUÁ RỰC. Hãy giảm bớt color_enhance_factor.")
        else:
            print("CHẨN ĐOÁN: Màu sắc đã đạt độ bão hòa tương đồng với ảnh gốc.")
    else:
        print("Không có dữ liệu.")

if __name__ == "__main__":
    folder_gt = "dataset/test"
    folder_out = "out_test_25000"
    calculate_full_metrics(folder_gt, folder_out)

## Demo Gradio

In [ ]:
!pip install gradio gradio_imageslider timm

In [ ]:
!python gradio_app_WIP.py